# This notebook first defines GEVs then smaples from them and    
# then uses Bayseian to calculate the percentiles and computes the bias
##### Author: Omid Emamjomehzadeh (https://www.omidemam.com/)
##### Supervisor: Dr. Omar Wani (https://engineering.nyu.edu/faculty/omar-wani)
##### Hydrologic Systems Group @NYU (https://www.omarwani.com/)

In [ ]:
#import libraries
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib as mpl
import pandas as pd
import numpy as np
import pymc3 as pm3
import scipy.stats as stats
from scipy import stats
from scipy.stats import genextreme
from scipy.stats import norm, gaussian_kde
from scipy.optimize import minimize
import seaborn as sns
import arviz as az
import pymc3 as pm
import os
import theano.tensor as tt
from theano.compile.ops import as_op
import theano
import pandas as pd
from pymc3.distributions.dist_math import bound
from scipy.stats import genextreme
import warnings
from arviz.plots import plot_utils as azpu
import arviz as az
from tqdm import tqdm
%matplotlib inline
sns.set()
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore', UserWarning)
theano.config.warn.round=False
from watermark import watermark
from datetime import datetime
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt
plt.style.use('default')   # <-- resets to plain Matplotlib look
from itertools import chain
import random


In [2]:
# read parameters
mle_1h = r"D:\BMM-IDF4Drainage_data_results\Percentile\gev_params_1h_mle_AORC.csv"
# Read the path
df_mle_1h = pd.read_csv(mle_1h)
df_mle_1h

,City,shape(c),loc,scale
0,Birmingham,0.072627,22.491624,10.228208
1,Phoenix,-0.267678,8.175461,3.664874
2,Little Rock,-0.055021,21.988082,9.258394
3,Los Angeles,0.042387,11.861270,5.693252
4,Denver,-0.413798,7.283690,4.342796
5,Hartford,-0.285448,14.298267,6.394461
6,Wilmington,-0.035867,18.485770,8.115728
7,Miami,-0.285025,19.471994,9.999978
8,Atlanta,-0.226416,16.665304,7.189982
9,Boise,-0.174572,5.606417,2.189396


In [3]:
import contextlib, os
import logging, warnings
# Quiet PyMC3/Theano/ArviZ info logs
for name in ("pymc3", "pymc", "theano", "aesara", "pytensor", "arviz"):
    logging.getLogger(name).setLevel(logging.ERROR)

# Turn off the progress bar from pm.sample
PM_PROGRESSBAR = dict(progressbar=False)

# Hide the specific divergence warnings
warnings.filterwarnings(
    "ignore",
    message=r".*divergences? after tuning.*",   # matches "There was/There were ... divergences after tuning."
    category=UserWarning,
)
# (Optional) also hide other sampling/diagnostic warnings from PyMC/ArviZ
warnings.filterwarnings("ignore", category=UserWarning, module=r".*pymc3|pymc.*")
warnings.filterwarnings("ignore", category=UserWarning, module=r".*arviz.*")

In [4]:
# Likelihood Function
def gev_logp(value,μ  ,  σ,  ξ):
        scaled = (value - μ ) /  σ
        # non-zero
        logp_xi_not_zero = -(tt.log( σ)
                 + (( ξ + 1) /  ξ) * tt.log1p( ξ * scaled)
                 + (1 +  ξ * scaled) ** (-1/ ξ))
        #zero
        logp_xi_zero = -tt.log( σ) + ( ξ+1)*(-(value - μ )/ σ) - tt.exp(-(value - μ )/ σ)
        #combined
        logp = tt.switch(tt.abs_( ξ) > 1e-4  , logp_xi_not_zero, logp_xi_zero) 
        return tt.sum(logp)

## NYC

In [ ]:
results = []
simseries_accum = [] 

i=29
city  =df_mle_1h.City[i]
c  =df_mle_1h['shape(c)'][i]
loc  =df_mle_1h['loc'][i]
scale  =df_mle_1h['scale'][i]

c_true, loc_true, scale_true = c, loc, scale

n_sims = 500
random.seed(12345)
seeds = random.sample(range(1, 100001), n_sims)

T = np.array([2, 5, 10, 25, 50, 100])
p = 1 - 1 / T
bias = np.zeros((n_sims, len(T)))

for i in range(n_sims):
    samples = genextreme.rvs(c=c_true, loc=loc_true, scale=scale_true, size=46, random_state=seeds[i])
    obs_samples = samples #no noise

    n_draws = 1000
    sample_number = 3500
    chain_n=4
    # initialize the arrays
    save_samples_24h = np.zeros([3, sample_number*chain_n])
    p_bayes_all_cities = np.zeros([ sample_number*chain_n,n_draws])
    percentiles_all_cities = np.zeros([ 6])
    percentiles = [50, 80, 90, 96, 98, 99]

    with pm.Model() as model_gev:
        μ = pm.Normal("μ", mu=0.1, sigma=100)
        σ = pm.Normal("σ", mu=0.1, sigma=100)
        ξ = pm.Normal("ξ", mu=0.15, sigma=100)

        gev_24h = pm.DensityDist('gev', gev_logp, observed={'value': obs_samples, 'μ': μ, 'σ': σ, 'ξ': ξ})
        step = pm.NUTS(target_accept=0.999,max_treedepth=15)
        trace_24h = pm.sample(sample_number, step=step, cores=4, chains=chain_n, tune=10_000, return_inferencedata=True, 
                            init="jitter+adapt_diag",
                            idata_kwargs={"density_dist_obs": False})


        stacked_24h = trace_24h.posterior.stack(draws=("chain", "draw"))
        
        shape_posterior1 = stacked_24h.ξ.values
        loc_posterior1 = stacked_24h.μ.values
        scale_posterior1 = stacked_24h.σ.values
        save_samples_24h[ 0, :] = shape_posterior1   
        save_samples_24h[ 1, :] = loc_posterior1 
        save_samples_24h[ 2, :] = scale_posterior1 
        c     = (-save_samples_24h[ 0, :])[:, None]                 
        loc   = ( save_samples_24h[ 1, :])[:, None]                 
        scale = (save_samples_24h[ 2, :])[:, None]
        
        # Draw 1000 samples for each of 3500*chain_n posterior parameter sets
        draws = genextreme.rvs(c=c, loc=loc, scale=scale,
                            size=(sample_number*chain_n, n_draws), random_state=12345)
        p_bayes_all_cities[ :, :] = draws
        # Percentiles over all 3500*chain_n*n_draws samples for city i
        percentiles_all_cities[ :] = np.percentile(draws.ravel(), percentiles)
        rl_true = genextreme.ppf(p, c=c_true, loc=loc_true, scale=scale_true)
        bias[i, :] = percentiles_all_cities[ :] - rl_true

        #save the bias
        sim_idx = np.arange(1, n_sims + 1)
        df_bias_city = pd.DataFrame({'City': city, 'sim': sim_idx})
        for j, t in enumerate(T):
            df_bias_city[f'bias_T{int(t)}'] = bias[:, j]
        df_bias_city.to_csv(rf"D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\bias_sim_series_{city}.csv", index=False)    



## All cities

In [ ]:
simseries_accum = []
# --- settings ---
n_sims = 500
random.seed(12345)
seeds = random.sample(range(1, 100001), n_sims)

T = np.array([2, 5, 10, 25, 50, 100])
p = 1 - 1 / T

sample_sizes = [46]   # last N years
AORC_END_YEAR = 2024              # "last N years" ends at 2024

for idx, row in df_mle_1h.iterrows():
    city       = row['City']
    c_true     = row['shape(c)']   # keep sign convention consistent with your genextreme usage
    loc_true   = row['loc']
    scale_true = row['scale']

    # true RLs once per city
    rl_true = genextreme.ppf(p, c=c_true, loc=loc_true, scale=scale_true)

    for sample_size in sample_sizes:

        # pre-alloc PER (CITY, SAMPLE_SIZE)
        bias   = np.zeros((n_sims, len(T)))

        for i in range( n_sims):  # 1-based loop
            # --- simulate one replicate with N = sample_size ---
            print(city+str(i))
            try:
                
                samples = genextreme.rvs(c=c_true, loc=loc_true, scale=scale_true, size=sample_size, random_state=seeds[i])
                obs_samples = samples # no noise

                n_draws = 1000
                sample_number = 3500
                chain_n=4
                # initialize the arrays
                save_samples_24h = np.zeros([3, sample_number*chain_n])
                p_bayes_all_cities = np.zeros([ sample_number*chain_n,n_draws])
                percentiles_all_cities = np.zeros([ 6])
                percentiles = [50, 80, 90, 96, 98, 99]

                with pm.Model() as model_gev:
                    μ = pm.Normal("μ", mu=0.1, sigma=100)
                    σ = pm.Normal("σ", mu=0.1, sigma=100)
                    ξ = pm.Normal("ξ", mu=0.15, sigma=100)

                    gev_24h = pm.DensityDist('gev', gev_logp, observed={'value': obs_samples, 'μ': μ, 'σ': σ, 'ξ': ξ})
                    step = pm.NUTS(target_accept=0.999,max_treedepth=15)
                    trace_24h = pm.sample(sample_number, step=step, cores=4, chains=chain_n, tune=10_000, return_inferencedata=True, jitter_max_retries=1000,
                                        init="jitter+adapt_diag",progressbar=False,
                                        idata_kwargs={"density_dist_obs": False})


                    stacked_24h = trace_24h.posterior.stack(draws=("chain", "draw"))
                    
                    shape_posterior1 = stacked_24h.ξ.values
                    loc_posterior1 = stacked_24h.μ.values
                    scale_posterior1 = stacked_24h.σ.values
                    save_samples_24h[ 0, :] = shape_posterior1   
                    save_samples_24h[ 1, :] = loc_posterior1 
                    save_samples_24h[ 2, :] = scale_posterior1 
                    c     = (-save_samples_24h[ 0, :])[:, None]                 
                    loc   = ( save_samples_24h[ 1, :])[:, None]                 
                    scale = (save_samples_24h[ 2, :])[:, None]
                    
                    # Draw 1000 samples for each of 3500*chain_n posterior parameter sets
                    draws = genextreme.rvs(c=c, loc=loc, scale=scale,
                                        size=(sample_number*chain_n, n_draws), random_state=12345)
                    p_bayes_all_cities[ :, :] = draws
                    # Percentiles over all 3500*chain_n*n_draws samples for city i
                    percentiles_all_cities[ :] = np.percentile(draws.ravel(), percentiles)
                    rl_true = genextreme.ppf(p, c=c_true, loc=loc_true, scale=scale_true)
                    bias[i, :] = percentiles_all_cities[ :] - rl_true

            except Exception:
                # if anything fails for this i, mark its bias as NaN and continue
                bias[i, :] = np.nan
                continue
            #save the bias
            sim_idx = np.arange(1, n_sims + 1)
            df_bias_city = pd.DataFrame({'City': city, 'sim': sim_idx})
            for j, t in enumerate(T):
                df_bias_city[f'bias_T{int(t)}'] = bias[:, j]
            df_bias_city.to_csv(rf"D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Stationary\bias_sim_series_{city}_N{sample_size}.csv", index=False)     # << fixed off-by-one


In [23]:
#Getting the current date and time
current_datetime = datetime.now()

# Printing the date and time
print("Date and Time of the Notebook Analysis:", current_datetime)

Date and Time of the Notebook Analysis: 2025-10-23 18:05:38.733360


In [24]:
%load_ext watermark

# Print the Python version and some dependencies
%watermark -v -m -p numpy,pandas,matplotlib,pymc3,scipy,seaborn,arviz,os,theano,warnings,tqdm,watermark


Python implementation: CPython
Python version       : 3.9.19
IPython version      : 8.18.1

numpy     : 1.22.1
pandas    : 2.0.3
matplotlib: 3.8.4
pymc3     : 3.11.6
scipy     : 1.7.3
seaborn   : 0.13.2
arviz     : 0.12.1
os        : unknown
theano    : 1.1.2
warnings  : unknown
tqdm      : 4.66.4
watermark : 2.4.3

Compiler    : MSC v.1929 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 24
Architecture: 64bit

